In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -qU langchain langchain-community langchain-core langchain-huggingface chromadb sentence-transformers
!pip install -qU transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66

In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 경로 설정
LOCAL_MODEL_PATH = "/content/drive/MyDrive/DILAB/Models/Qwen2-7B-Instruct"

# 벡터 DB가 저장된 경로
BASE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment"
DB_PATH = os.path.join(BASE_DIR, "VectorDB")

# 2. 벡터 DB 로드
print(f"[백터 DB 경로]: {DB_PATH}")

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print("벡터 DB 로드 완료")

# 3. 로컬 Qwen 모델 로드
print(f"[로컬 모델 경로]: {LOCAL_MODEL_PATH}")

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"{LOCAL_MODEL_PATH}")
else:
    try:
        # 1. 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)

        # 2. 양자화 설정 (4bit 로딩을 위한 설정 객체 생성)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        # 3. 모델 로드
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL_PATH,
            device_map="auto",
            quantization_config=bnb_config,
            dtype=torch.float16
        )

        # 4. 파이프라인 구축
        text_generator = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=512,
            temperature=0.1
        )
        print("Qwen2-7B-Instruct 모델 로드 완료")

    except Exception as e:
        print(e)

[백터 DB 경로]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

/tmp/ipython-input-2137132214.py:23: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)


벡터 DB 로드 완료
[로컬 모델 경로]: /content/drive/MyDrive/DILAB/Models/Qwen2-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Qwen2-7B-Instruct 모델 로드 완료


In [6]:
import warnings
from transformers import logging

# 0. 경고 메시지 차단
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

def experiment_rag_comparison(question):
    print(f"\n{'='*80}")
    print(f"[질문]: {question}")
    print(f"{'='*80}\n")

    # [Case 1] 순수 Qwen (RAG 미적용)
    print("[1. RAG 적용 전 (Original Qwen)]")

    messages_no_rag = [
        {"role": "system", "content": "당신은 e스포츠 전문가입니다. 질문에 답변해 주세요."},
        {"role": "user", "content": question}
    ]
    prompt_no_rag = tokenizer.apply_chat_template(messages_no_rag, tokenize=False, add_generation_prompt=True)

    output_no_rag = text_generator(
        prompt_no_rag,
        max_new_tokens=512,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_no_rag[0]['generated_text'].strip()}")
    print(f"\n{'-'*80}\n")

    # [Case 2] RAG 적용
    print("[2. RAG 적용 후]")

    raw_docs = vector_db.similarity_search(question, k=10) # 넉넉히 10개 가져옴
    docs = []
    seen_titles = set()

    for doc in raw_docs:
        title = doc.metadata.get("title", "제목 없음")
        if title in seen_titles:
            continue
        seen_titles.add(title)
        docs.append(doc)
        if len(docs) >= 5:
            break

    context_text = ""
    print(f"\n[검색된 근거 자료]")
    if not docs:
        print("검색된 문서 없음")
    else:
        for i, doc in enumerate(docs):
            title = doc.metadata.get("title", "제목 없음")
            date = doc.metadata.get("date", "날짜 미상")
            preview = doc.page_content.replace("\n", " ").strip()[:80]
            print(f"      [{i+1}] {title} ({date}) -> {preview}...")
            context_text += f"문서{i+1}: {doc.page_content}\n\n"

    # 2. 프롬프트 구성
    system_prompt = f"""
    당신은 e스포츠 분석가입니다.
    아래 [참고 자료]에 있는 내용만을 근거로 질문에 답변하세요.

    [지침]
    1. '참고 자료'에 없는 내용은 절대 지어내지 마세요.
    2. 질문과 관련 없는 내용은 배제하고 핵심만 답변하세요.

    [참고 자료]
    {context_text}
    """

    messages_rag = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompt_rag = tokenizer.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)

    # 3. 실제 입력된 프롬프트 출력
    print(f"\n[LLM에게 실제로 들어가는 입력 데이터 (Prompt)]")
    print(f"   {prompt_rag[:300]} \n   ... (중략: 기사 본문들) ... \n   {prompt_rag[-200:]}")

    # 4. 답변 생성
    output_rag = text_generator(
        prompt_rag,
        max_new_tokens=1024,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_rag[0]['generated_text'].strip()}")
    print(f"\n{'='*80}")

# 실행
experiment_rag_comparison("2026년 LCK CUP에서 DRX의 전적을 알려줘")


[질문]: 2026년 LCK CUP에서 DRX의 전적을 알려줘

[1. RAG 적용 전 (Original Qwen)]

답변:
죄송합니다, 하지만 현재 시점에서 2026년 LCK 챔피언십(CUP)에서 DRX의 전적을 알 수 없습니다. 이는 미래의 이벤트이며, 현재의 데이터나 정보로는 예측할 수 없습니다. 또한, 스포츠 경기 결과는 여러 변수에 의해 결정되므로, 정확한 전적을 알려드리는 것은 불가능합니다. 

그러나 DRX의 성과를 예측하는 데 도움이 될 수 있는 몇 가지 요소를 고려하실 수 있습니다. 예를 들어, 팀의 선수들의 능력, 팀워크, 전략, 경기 경험, 그리고 경기 상대의 강점을 고려하면 어느 정도 추론이 가능할 수 있습니다. 그러나 이러한 정보는 미래의 확실한 결과를 보장하지는 않습니다.

또한, 스포츠 경기의 경우, 시즌 동안 팀의 성적은 변동될 수 있으며, 이는 팀의 성장, 선수들의 상태 변화, 또는 경쟁 팀의 강화와 같은 다양한 요인에 의해 영향을 받습니다. 따라서, 최종적인 전적은 시즌 동안의 여러 사건과 상황에 따라 달라질 수 있습니다.

--------------------------------------------------------------------------------

[2. RAG 적용 후]

[검색된 근거 자료]
      [1] 조재읍 “당장의 승패에 연연하지 않겠다” (2026-01-21) -> 내용: DRX 조재읍 감독이 당장의 승패에 연연하지 않고 멀리 내다보겠다고 말했다. DRX는 21일 서울 종로구 LCK 아레나에서 열린 2026...
      [2] 2026 LCK컵, '체급이 다르다' 증명한 DK... 브리온 54분 셧아웃, 장로 그룹 2연승 질주 (2026-01-14) -> 제목: 2026 LCK컵, '체급이 다르다' 증명한 DK... 브리온 54분 셧아웃, 장로 그룹 2연승 질주...
      [3] PO 직행·탈락 가르는 승부처…2026 LCK컵 3주 차 관전 포인트는 (2026-01-24) -> 

In [7]:
!pip install -qU langchain langchain-community langchain-core rank_bm25 sentence-transformers chromadb

In [8]:
import warnings
from transformers import logging
import numpy as np
import os

# 1. 경고 메시지 끄기
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# 2. 필수 라이브러리 임포트
try:
    from langchain_community.retrievers import BM25Retriever
    from sentence_transformers import CrossEncoder
    from langchain_core.documents import Document
except ImportError:
    os.system("pip install -qU langchain-community rank_bm25 sentence-transformers chromadb")
    from langchain_community.retrievers import BM25Retriever
    from sentence_transformers import CrossEncoder
    from langchain_core.documents import Document

# 3. 리랭커 모델 로드
if 'reranker_model' not in globals():
    reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3', automodel_args={'torch_dtype': 'auto'})
else:
    print("리랭커 모델이 이미 로드되어 있습니다.")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [10]:
# [앙상블 검색 + 리랭킹 함수]
def advanced_search(question, k_final=5):
    # 1. 벡터 검색 (의미 기반) - 10개
    vector_docs = vector_db.similarity_search(question, k=10)

    # 2. BM25 검색 (키워드 기반) - 10개
    # (매 검색마다 DB에서 텍스트를 가져와서 BM25를 만듭니다)
    all_data = vector_db.get() # 전체 데이터 가져오기
    bm25_docs = [Document(page_content=t, metadata=m)
                 for t, m in zip(all_data['documents'], all_data['metadatas'])]

    bm25_retriever = BM25Retriever.from_documents(bm25_docs)
    bm25_retriever.k = 10
    keyword_docs = bm25_retriever.invoke(question)

    # 3. 결과 합치기 & 중복 제거
    combined_docs = vector_docs + keyword_docs

    # 내용이 같으면 제거
    unique_docs = {}
    for doc in combined_docs:
        unique_docs[doc.page_content] = doc

    candidate_docs = list(unique_docs.values())

    if not candidate_docs:
        return []

    # 4. 리랭킹
    pairs = [[question, doc.page_content] for doc in candidate_docs]
    scores = reranker_model.predict(pairs)

    # 점수 높은 순 정렬
    sorted_indices = np.argsort(scores)[::-1]

    final_docs = []
    print(f"\n[통합 검색 및 리랭킹 결과 상위 {k_final}개]")

    for i in range(min(k_final, len(candidate_docs))):
        idx = sorted_indices[i]
        doc = candidate_docs[idx]
        score = scores[idx]
        final_docs.append(doc)

        # 디버깅 출력
        title = doc.metadata.get('title', '제목없음')
        print(f"   [{i+1}등] 점수: {score:.4f} | {title}")

    return final_docs

# 최종 질문 함수
def ask_smart_rag(question):
    print(f"\n{'='*80}")
    print(f"[RAG 질문]: {question}")

    relevant_docs = advanced_search(question, k_final=5)

    if not relevant_docs:
        print("관련된 문서를 찾지 못했습니다.")
        return

    # 2. 문맥 만들기
    context_text = ""
    for i, doc in enumerate(relevant_docs):
        context_text += f"문서{i+1}: {doc.page_content}\n\n"

    # 3. 프롬프트 작성
    system_prompt = f"""
    당신은 팩트 기반 e스포츠 분석가입니다.
    제공된 [문서]를 정밀하게 분석하여 질문에 답하세요.

    [지침]
    1. 질문에 특정 날짜(예: 1월 23일)가 있다면, 문서 내의 '날짜' 정보와 정확히 대조하세요.
    2. 경기 결과(승패, 스코어)는 반드시 문서에 명시된 내용만 말하세요.
    3. 문서에 정답이 없다면 솔직하게 "문서에 해당 날짜의 경기 정보가 없습니다"라고 답하세요.

    [문서]
    {context_text}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 4. 답변 생성
    output = text_generator(prompt, max_new_tokens=1024, max_length=None, return_full_text=False)
    print(f"\n[답변]:\n{output[0]['generated_text'].strip()}")
    print(f"{'='*80}")

ask_smart_rag("2026년 LCK CUP에서 펜타킬을 한 선수를 알려줘")


[RAG 질문]: 2026년 LCK CUP에서 펜타킬을 한 선수를 알려줘

[통합 검색 및 리랭킹 결과 상위 5개]
   [1등] 점수: 0.8527 | [LCK컵] 졌다고 생각한 순간, 에이밍이 웃었다... '펜타킬 반전극' KT, 브리온 격파
   [2등] 점수: 0.6532 | 젠지·T1, ‘2강 체제’ 굳히나…‘디펜딩 챔피언’ 한화생명, LCK컵 2연패 충격 [SS시선집중]
   [3등] 점수: 0.3047 | [LCK] 컵 대회 2주차, T1 vs KT 롤스터 ’월드 챔피언십 결승전 리매치’ 진행
   [4등] 점수: 0.2722 | 2026 LCK컵 2주 차, T1-kt 롤스터 맞대결… ‘롤드컵 결승 리매치’ 성사
   [5등] 점수: 0.2573 | PO 직행·탈락 가르는 승부처…2026 LCK컵 3주 차 관전 포인트는

[답변]:
문서1에서 2026년 LCK CUP에서 펜타킬을 한 선수로는 '에이밍' 김하람이 언급되어 있습니다.
